In [1]:
# ===================================================================
# Lightweight Vehicle Re-identification with MobileNetV2 and 640x640 Images
# NEW APPROACH: Train as a Classifier, Test as a Metric Model
# ===================================================================

import os
import random
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.model_selection import KFold
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import json
import shutil # Added for cleanup

# --- 1. CONFIGURATION ---
# IMPORTANT: Change this path to where your dataset is located
BASE_DATASET_PATH = "/kaggle/input/nic-similarity-vehicle/AVD_structural_similarity/AVD_structural_similarity"

# Model & Training Parameters
IMAGE_SIZE = (640, 640)
BATCH_SIZE = 8
EPOCHS_INITIAL = 15
EPOCHS_FINE_TUNE = 5
LEARNING_RATE_INITIAL = 1e-4
LEARNING_RATE_FINE_TUNE = 1e-5
EMBEDDING_DIM = 512
MARGIN = 0.5  # No longer used by main loss, but kept for reference

# Cross-validation parameters
N_FOLDS = 5
TEST_SPLIT = 0.1  # 20% of data reserved for final testing

# --- 2. ADVANCED MODEL COMPONENTS (CUSTOM KERAS LAYERS) ---

# ... (ChannelAttention, SpatialAttention, GrayScale classes remain exactly the same) ...
class ChannelAttention(layers.Layer):
    def __init__(self, ratio=8, **kwargs):
        super(ChannelAttention, self).__init__(**kwargs)
        self.ratio = ratio
        
    def build(self, input_shape):
        channel = input_shape[-1]
        self.shared_mlp_one = layers.Dense(channel // self.ratio, activation='relu', use_bias=False)
        self.shared_mlp_two = layers.Dense(channel, use_bias=False)
        super(ChannelAttention, self).build(input_shape)
        
    def call(self, inputs):
        avg_pool = layers.GlobalAveragePooling2D()(inputs)
        avg_pool = layers.Reshape((1, 1, avg_pool.shape[-1]))(avg_pool)
        max_pool = layers.GlobalMaxPooling2D()(inputs)
        max_pool = layers.Reshape((1, 1, max_pool.shape[-1]))(max_pool)
        mlp_avg = self.shared_mlp_two(self.shared_mlp_one(avg_pool))
        mlp_max = self.shared_mlp_two(self.shared_mlp_one(max_pool))
        channel_attention = layers.Activation('sigmoid')(layers.Add()([mlp_avg, mlp_max]))
        return inputs * channel_attention

class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super(SpatialAttention, self).__init__(**kwargs)
        self.kernel_size = kernel_size
        
    def build(self, input_shape):
        self.conv = layers.Conv2D(1, kernel_size=self.kernel_size, padding='same', activation='sigmoid')
        super(SpatialAttention, self).build(input_shape)
        
    def call(self, inputs):
        avg_pool = layers.Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(inputs)
        max_pool = layers.Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(inputs)
        concat = layers.Concatenate(axis=-1)([avg_pool, max_pool])
        spatial_attention = self.conv(concat)
        return inputs * spatial_attention

class GrayScale(layers.Layer):
    def __init__(self, **kwargs):
        super(GrayScale, self).__init__(**kwargs)
        
    def call(self, inputs):
        return tf.image.rgb_to_grayscale(inputs)


# --- 3. DATA PREPROCESSING ---

# ... (gray_world_correction, read_image functions remain exactly the same) ...
def gray_world_correction(img):
    """Apply Gray World color constancy correction."""
    mean_b = np.mean(img[:,:,0])
    mean_g = np.mean(img[:,:,1])
    mean_r = np.mean(img[:,:,2])
    gray_mean = (mean_b + mean_g + mean_r) / 3
    
    if mean_b == 0: mean_b = 1
    if mean_g == 0: mean_g = 1
    if mean_r == 0: mean_r = 1

    scale_b = gray_mean / mean_b
    scale_g = gray_mean / mean_g
    scale_r = gray_mean / mean_r
    
    img[:,:,0] = img[:,:,0] * scale_b
    img[:,:,1] = img[:,:,1] * scale_g
    img[:,:,2] = img[:,:,2] * scale_r
    
    return np.clip(img, 0, 255).astype(np.uint8)

def read_image(path, target_size=IMAGE_SIZE, apply_clahe=True, augment=False):
    """Read and preprocess an image with optional augmentation."""
    img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Could not read image at path: {path}")
        
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = gray_world_correction(img)
    
    if apply_clahe:
        hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        h, s, v = cv2.split(hsv)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        v = clahe.apply(v)
        hsv = cv2.merge([h, s, v])
        img = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

    if augment:
        if random.random() > 0.5:
            img = cv2.flip(img, 1)
        if random.random() > 0.5:
            brightness_factor = 0.8 + random.random() * 0.4
            img = np.clip(img * brightness_factor, 0, 255)
        if random.random() > 0.7:
            kernel_size = random.choice([3, 5])
            img = cv2.GaussianBlur(img, (kernel_size, kernel_size), 0)

    img = cv2.resize(img, target_size[::-1], interpolation=cv2.INTER_AREA)
    img = img.astype(np.float32) / 255.0
    return img


# --- 4. LOSS FUNCTIONS ---

# ... (All loss functions - triplet_loss, improved_contrastive_loss, circle_loss - are no longer needed for training) ...
# ... (You can keep them if you want, but they are not used by the new model) ...


# --- 5. MODEL BUILDING FUNCTIONS ---

def cbam_block(feature_map, ratio=8):
    """Convolutional Block Attention Module (CBAM)"""
    channel_refined = ChannelAttention(ratio=ratio)(feature_map)
    refined = SpatialAttention()(channel_refined)
    return refined

def color_branch(input_tensor):
    """Branch to extract color-specific features"""
    x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(input_tensor)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    return x

def texture_branch(input_tensor):
    """Branch to extract texture-specific features from grayscale"""
    gray = GrayScale()(input_tensor)
    x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(gray)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    return x

### --- NEW MODEL FUNCTION --- ###
def build_classification_model(input_shape=(640,640,3), num_vehicle_ids=1000):
    """
    Builds the Re-ID model for Classification Training.
    
    This function creates and returns three things:
    1. training_model: Takes a single image and outputs classification logits.
                       This is *only* for training.
    2. inference_model: Takes a single image and outputs its L2-normalized
                        embedding. This is *only* for testing/evaluation.
    3. base_cnn: The MobileNetV2 backbone (for fine-tuning).
    """
    
    # --- 1. Define the Shared Architecture ---
    img_input = layers.Input(input_shape, name="img_input")
    
    # Base CNN (MobileNetV2)
    base_cnn = MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape, name='mobilenetv2_base')
    # Freeze initial layers
    for layer in base_cnn.layers[:-30]:
        layer.trainable = False
    
    # Main branch
    x = base_cnn(img_input)
    x = cbam_block(x)
    x = layers.GlobalAveragePooling2D()(x)
    main_emb = layers.Dense(EMBEDDING_DIM, activation=None)(x)
    
    # Color and Texture branches
    color_emb = color_branch(img_input)
    texture_emb = texture_branch(img_input)
    
    # Combine all features
    combined_emb = layers.Concatenate()([main_emb, color_emb, texture_emb])
    
    # Final shared feature layer (this is our embedding *before* normalization)
    final_emb_features = layers.Dense(EMBEDDING_DIM, activation='relu', name='final_emb_features')(combined_emb)
    
    # --- 2. Define the Two Heads ---
    
    # HEAD 1: Metric Learning (Embedding) - L2 Normalized
    # This head is *only* used for the inference_model
    normalized_emb = layers.Lambda(
        lambda x: tf.math.l2_normalize(x, axis=1), 
        name='metric_output'
    )(final_emb_features)
    
    # HEAD 2: ID Classification (Logits)
    # This head is *only* used for the training_model
    id_output = layers.Dense(
        num_vehicle_ids, 
        activation='softmax', # Softmax for categorical cross-entropy
        name='id_output'
    )(final_emb_features)
    
    # --- 3. Create the Models ---
    
    # The TRAINING model: Input -> Image, Output -> ID Logits
    training_model = models.Model(
        inputs=img_input, 
        outputs=id_output, 
        name='classification_training_model'
    )
    
    # The INFERENCE model: Input -> Image, Output -> Normalized Embedding
    inference_model = models.Model(
        inputs=img_input,
        outputs=normalized_emb, 
        name='inference_model'
    )
    
    # Return all three components
    return training_model, inference_model, base_cnn


# --- 6. DATA GENERATORS ---

### --- NEW GENERATOR --- ###
class IDClassificationGenerator(tf.keras.utils.Sequence):
    """
    Data generator for Classification-Only (ID) training.
    Yields (image, one_hot_id_label) pairs.
    """
    def __init__(self, entry_folder, exit_folder, train_images, batch_size=BATCH_SIZE, augment=True, **kwargs):
        super(IDClassificationGenerator, self).__init__(**kwargs)
        self.entry_folder = entry_folder
        self.exit_folder = exit_folder
        self.batch_size = batch_size
        self.augment = augment
        
        # Create Vehicle ID mapping
        self.vehicle_ids = sorted(list(set(train_images)))
        self.num_vehicle_ids = len(self.vehicle_ids)
        self.id_to_index = {vid: i for i, vid in enumerate(self.vehicle_ids)}
        
        # Create a list of all available samples (image_path, label_index)
        self.all_samples = []
        for img_name in self.vehicle_ids:
            label_index = self.id_to_index[img_name]
            
            entry_path = os.path.join(self.entry_folder, img_name)
            exit_path = os.path.join(self.exit_folder, img_name)
            
            if os.path.exists(entry_path):
                self.all_samples.append((entry_path, label_index))
            if os.path.exists(exit_path):
                self.all_samples.append((exit_path, label_index))
        
        self.on_epoch_end()
        print(f"IDClassificationGenerator found {len(self.all_samples)} training samples for {self.num_vehicle_ids} IDs.")

    def __len__(self):
        return int(np.ceil(len(self.all_samples) / self.batch_size))
        
    def __getitem__(self, idx):
        # Get batch of sample info
        batch_samples = self.all_samples[idx * self.batch_size : (idx + 1) * self.batch_size]
        
        # Initialize batch arrays
        X_batch = np.zeros((len(batch_samples), *IMAGE_SIZE, 3))
        y_batch = np.zeros((len(batch_samples), self.num_vehicle_ids))
        
        for i, (img_path, label_index) in enumerate(batch_samples):
            # Load and augment image
            X_batch[i] = read_image(img_path, augment=self.augment)
            
            # Create one-hot label
            y_batch[i, label_index] = 1.0
            
        return X_batch, y_batch

    def on_epoch_end(self):
        """Shuffles indexes after each epoch"""
        random.shuffle(self.all_samples)


# --- 7. EVALUATION FUNCTIONS ---

# ... (calculate_ssim, create_test_pairs, evaluate_model_with_ssim, precision_recall_curve_support functions remain exactly the same) ...
# Note: evaluate_model_with_ssim will work perfectly with the new `inference_model`
# because it just expects a model that takes one image and returns an embedding.
def calculate_ssim(img1, img2):
    """Calculate Structural Similarity Index between two images."""
    gray1 = cv2.cvtColor((img1 * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    gray2 = cv2.cvtColor((img2 * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    score, _ = ssim(gray1, gray2, data_range=255, full=True)
    return score

def create_test_pairs(entry_images, exit_images):
    """Create positive and negative pairs for testing."""
    exit_image_map = {img: i for i, img in enumerate(exit_images)}
    
    positive_pairs = []
    for i, entry_img in enumerate(entry_images):
        if entry_img in exit_image_map:
            positive_pairs.append((i, exit_image_map[entry_img], 1))
    
    negative_pairs = []
    for i, entry_img in enumerate(entry_images):
        possible_negatives = [j for j, exit_img in enumerate(exit_images) if exit_img != entry_img]
        if possible_negatives:
            exit_idx = random.choice(possible_negatives)
            negative_pairs.append((i, exit_idx, 0))
    
    return positive_pairs, negative_pairs

def evaluate_model_with_ssim(embedding_model, entry_folder, exit_folder, test_pairs):
    """Evaluates model with SSIM and other metrics."""
    y_true, y_scores, ssim_scores = [], [], []
    
    entry_images = sorted([f for f in os.listdir(entry_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    exit_images = sorted([f for f in os.listdir(exit_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    
    print("Evaluating model on test pairs...")
    for entry_idx, exit_idx, label in tqdm(test_pairs, desc="Evaluating"):
        entry_name = entry_images[entry_idx]
        exit_name = exit_images[exit_idx]
        
        entry_path = os.path.join(entry_folder, entry_name)
        exit_path = os.path.join(exit_folder, exit_name)
        
        img1 = read_image(entry_path, augment=False)
        img2 = read_image(exit_path, augment=False)
        
        ssim_score = calculate_ssim(img1, img2)
        ssim_scores.append(ssim_score)
        
        img1_batch = np.expand_dims(img1, axis=0)
        img2_batch = np.expand_dims(img2, axis=0)
        
        emb1 = embedding_model.predict(img1_batch, verbose=0)[0]
        emb2 = embedding_model.predict(img2_batch, verbose=0)[0]
        
        distance = 1 - np.dot(emb1, emb2)
        
        y_true.append(label)
        y_scores.append(distance)

    y_scores_inverted = 1 - np.array(y_scores)
    
    fpr, tpr, thresholds = roc_curve(y_true, y_scores_inverted)
    roc_auc = auc(fpr, tpr)
    
    youden_j = tpr - fpr
    optimal_idx = np.argmax(youden_j)
    optimal_similarity_threshold = thresholds[optimal_idx]
    
    optimal_distance_threshold = 1 - optimal_similarity_threshold 
    
    y_pred = (np.array(y_scores) < optimal_distance_threshold).astype(int)
    accuracy = np.mean(y_true == y_pred) * 100
    
    precision, recall, f1, _ = precision_recall_curve_support(y_true, y_pred, average='binary')
    
    pos_ssim = [s for s, t in zip(ssim_scores, y_true) if t == 1]
    neg_ssim = [s for s, t in zip(ssim_scores, y_true) if t == 0]
    
    print("\n📊 --- Evaluation Summary ---")
    print(f"✅ Total pairs tested: {len(y_true)} (Pos: {sum(y_true)}, Neg: {len(y_true) - sum(y_true)})")
    print(f"✅ Accuracy: {accuracy:.2f}%")
    print(f"✅ Precision: {precision:.4f}")
    print(f"✅ Recall: {recall:.4f}")
    print(f"✅ F1 Score: {f1:.4f}")
    print(f"✅ Average Positive Distance: {np.mean([s for s, t in zip(y_scores, y_true) if t == 1]):.4f}")
    print(f"✅ Average Negative Distance: {np.mean([s for s, t in zip(y_scores, y_true) if t == 0]):.4f}")
    print(f"✅ Optimal Distance Threshold: {optimal_distance_threshold:.4f}")
    print(f"✅ AUC: {roc_auc:.4f}")
    print(f"✅ Average SSIM for Positive Pairs: {np.mean(pos_ssim):.4f}")
    print(f"✅ Average SSIM for Negative Pairs: {np.mean(neg_ssim):.4f}")

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.scatter(fpr[optimal_idx], tpr[optimal_idx], marker='o', color='red', 
                label=f'Optimal threshold @ {optimal_similarity_threshold:.4f} (similarity)')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc="lower right")
    plt.savefig('roc_curve.png')
    print("✅ ROC curve saved as 'roc_curve.png'")
    plt.close()
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': roc_auc,
        'optimal_threshold': optimal_distance_threshold,
        'avg_pos_distance': np.mean([s for s, t in zip(y_scores, y_true) if t == 1]),
        'avg_neg_distance': np.mean([s for s, t in zip(y_scores, y_true) if t == 0]),
        'avg_pos_ssim': np.mean(pos_ssim),
        'avg_neg_ssim': np.mean(neg_ssim)
    }

def precision_recall_curve_support(y_true, y_pred, average='binary'):
    """Calculate precision, recall, and F1 score."""
    from sklearn.metrics import precision_score, recall_score, f1_score
    precision = precision_score(y_true, y_pred, average=average, zero_division=0)
    recall = recall_score(y_true, y_pred, average=average, zero_division=0)
    f1 = f1_score(y_true, y_pred, average=average, zero_division=0)
    return precision, recall, f1, None


# --- 8. TRAINING AND TESTING ---

def split_data_for_test(entry_folder, exit_folder, test_split=0.2):
    """Split data for training and testing."""
    entry_images_all = sorted([f for f in os.listdir(entry_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    exit_images_all = sorted([f for f in os.listdir(exit_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    
    common_images = sorted(list(set(entry_images_all) & set(exit_images_all)))
    
    random.shuffle(common_images)
    test_size = int(len(common_images) * test_split)
    test_images = common_images[:test_size]
    train_images = common_images[test_size:]
    
    entry_images = sorted([f for f in os.listdir(entry_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    exit_images = sorted([f for f in os.listdir(exit_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    
    entry_map = {name: i for i, name in enumerate(entry_images)}
    exit_map = {name: i for i, name in enumerate(exit_images)}

    positive_test_pairs = []
    for img_name in test_images:
        if img_name in entry_map and img_name in exit_map:
            positive_test_pairs.append((entry_map[img_name], exit_map[img_name], 1))
            
    negative_test_pairs = []
    for img_name in test_images:
        if img_name not in entry_map: continue
        
        entry_idx = entry_map[img_name]
        possible_negatives = [j for j, exit_img in enumerate(exit_images) if exit_img != img_name]
        if possible_negatives:
            exit_idx = random.choice(possible_negatives)
            negative_test_pairs.append((entry_idx, exit_idx, 0))
    
    test_pairs = positive_test_pairs + negative_test_pairs
    
    return train_images, test_pairs, entry_images, exit_images

### --- MODIFIED TRAINING FUNCTION --- ###
def train_and_test_model(entry_folder, exit_folder):
    """
    Train model as a classifier and test on positive/negative pairs.
    """
    # Split data
    train_images, test_pairs, entry_images, exit_images = split_data_for_test(entry_folder, exit_folder, TEST_SPLIT)
    
    print(f"Total common images: {len(train_images) + len([p for p in test_pairs if p[2] == 1])}")
    print(f"Training images (Unique IDs): {len(train_images)}")
    print(f"Test pairs: {len(test_pairs)} (Pos: {len([p for p in test_pairs if p[2] == 1])}, Neg: {len([p for p in test_pairs if p[2] == 0])})")
    
    # Create a temporary directory with only training images
    temp_entry_dir = "temp_entry"
    temp_exit_dir = "temp_exit"
    
    os.makedirs(temp_entry_dir, exist_ok=True)
    os.makedirs(temp_exit_dir, exist_ok=True)
    
    print("Copying training files to temporary directory...")
    for img_name in tqdm(train_images):
        entry_src = os.path.join(entry_folder, img_name)
        exit_src = os.path.join(exit_folder, img_name)
        
        entry_dst = os.path.join(temp_entry_dir, img_name)
        exit_dst = os.path.join(temp_exit_dir, img_name)
        
        if os.path.exists(entry_src):
             with open(entry_src, 'rb') as f_src, open(entry_dst, 'wb') as f_dst:
                f_dst.write(f_src.read())
        if os.path.exists(exit_src):
            with open(exit_src, 'rb') as f_src, open(exit_dst, 'wb') as f_dst:
                f_dst.write(f_src.read())
    
    # --- Create data generator and model ---
    
    # Get the dynamic number of unique vehicle IDs
    num_vehicle_ids = len(train_images)
    print(f"Dynamic num_vehicle_ids set to: {num_vehicle_ids}")
    
    # 1. Create the Generator
    train_gen = IDClassificationGenerator(
        temp_entry_dir, 
        temp_exit_dir, 
        train_images=train_images, 
        augment=True,
        batch_size=BATCH_SIZE
    )
    
    # 2. Build the Models
    # model = for training
    # embedding_model = for testing/inference
    model, embedding_model, base_cnn = build_classification_model(
        input_shape=IMAGE_SIZE + (3,),
        num_vehicle_ids=num_vehicle_ids
    )
    
    # 3. Compile the TRAINING model
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE_INITIAL), 
        loss='categorical_crossentropy',
        metrics=['accuracy'] # Monitor training accuracy
    )
        
    print("\n--- Training Model Summary ---")
    model.summary()
    print("\n--- Inference Model Summary ---")
    embedding_model.summary()
    
    # Callbacks (Monitor 'loss' since we don't have a validation split)
    checkpoint = ModelCheckpoint('best.weights.h5', monitor='loss', save_best_only=True, save_weights_only=True, verbose=1)
    early_stopping = EarlyStopping(monitor='loss', patience=5, verbose=1, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.1, patience=3, verbose=1, min_lr=1e-7)
    
    # --- Initial training ---
    print("\n--- Starting Initial Training (Training as Classifier) ---")
    history = model.fit(
        train_gen,
        epochs=EPOCHS_INITIAL,
        callbacks=[checkpoint, early_stopping, reduce_lr]
    )
    
    # --- Fine-tuning ---
    print("\n--- Starting Fine-Tuning (Training as Classifier) ---")
    
    # Unfreeze the last 30 layers of the base CNN
    print(f"Unfreezing last 30 layers of {base_cnn.name}...")
    for layer in base_cnn.layers[-30:]:
        layer.trainable = True
    print("Layers unfrozen.")
        
    # Re-compile with fine-tune learning rate
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE_FINE_TUNE), 
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    history_fine = model.fit(
        train_gen,
        epochs=EPOCHS_FINE_TUNE,
        callbacks=[checkpoint, early_stopping, reduce_lr],
        initial_epoch=history.epoch[-1] # Continue epoch count
    )
    
    # --- Evaluation ---
    
    # Load best weights into the TRAINING model
    print("Loading best weights from 'best.weights.h5'...")
    model.load_weights('best.weights.h5')
    
    # By loading weights into 'model', we have *also* loaded them
    # into 'embedding_model' because they share all the same layers.
    
    # Save the final INFERENCE model
    embedding_model.save("best_vehicle_reid_embedding_model.h5")
    print("\n✅ Best Embedding (Inference) model saved as 'best_vehicle_reid_embedding_model.h5'")
    
    # Evaluate on test set using the INFERENCE model
    print("\n--- Evaluating Model on Test Set (Using Embeddings) ---")
    test_results = evaluate_model_with_ssim(embedding_model, entry_folder, exit_folder, test_pairs)
    
    # Save results to JSON
    with open('test_results.json', 'w') as f:
        json.dump(test_results, f, indent=4)
    
    print("\n✅ Test results saved to 'test_results.json'")
    
    # Clean up temporary directories
    print("Cleaning up temporary directories...")
    shutil.rmtree(temp_entry_dir)
    shutil.rmtree(temp_exit_dir)
    
    return embedding_model, test_results

# ===================================================================
# --- 9. MAIN EXECUTION ---
# ===================================================================

# Set paths based on your dataset structure
entry_folder = os.path.join(BASE_DATASET_PATH, "Entry_processed")
exit_folder = os.path.join(BASE_DATASET_PATH, "Exit_processed")

# Sanity check
if not os.path.exists(entry_folder) or not os.path.exists(exit_folder):
    print(f"❌ Error: Folders not found!")
    print(f"    Please check BASE_DATASET_PATH: {BASE_DATASET_PATH}")
    print(f"    Expected Entry folder: {entry_folder}")
    print(f"    Expected Exit folder: {exit_folder}")
else:
    print("✅ Dataset folders found. Starting training process...")
    
    # --- MODIFIED: Removed comparison, just run the one new method ---
    
    print("\n🚀 Training with Classification Loss...")
    embedding_model, test_results = train_and_test_model(entry_folder, exit_folder)
    
    print("\n🎉 Classification-based Training completed successfully!")
    print(f"Test Accuracy: {test_results['accuracy']:.2f}%")
    print(f"Test AUC: {test_results['auc']:.4f}")
    print(f"Test Average SSIM for Positive Pairs: {test_results['avg_pos_ssim']:.4f}")
    print(f"Test Average SSIM for Negative Pairs: {test_results['avg_neg_ssim']:.4f}")

    print("\n--- All Done ---")

2025-11-11 07:58:00.275826: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762847880.472904      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762847880.535246      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 8.0756e-04 - loss: 6.2377
Epoch 1: loss improved from inf to 6.23549, saving model to best.weights.h5
120/120 ━━━━━━━━━━━━━━━━━━━━ 215s 2s/step - accuracy: 8.0949e-04 - loss: 6.2377 - learning_rate: 1.0000e-04
Epoch 2/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.0130 - loss: 6.0573
Epoch 2: loss improved from 6.23549 to 6.04373, saving model to best.weights.h5
120/120 ━━━━━━━━━━━━━━━━━━━━ 181s 2s/step - accuracy: 0.0130 - loss: 6.0571 - learning_rate: 1.0000e-04
Epoch 3/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.0750 - loss: 5.6574
Epoch 3: loss improved from 6.04373 to 5.59478, saving model to best.weights.h5
120/120 ━━━━━━━━━━━━━━━━━━━━ 191s 2s/step - accuracy: 0.0748 - loss: 5.6569 - learning_rate: 1.0000e-04
Epoch 4/15
120/120 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1389 - loss: 4.7541
Epoch 4: loss improved from 5.59478 to 4.62759, saving model to best.weights.h5
120/120 ━━━━━━━━━━━━━━━━━━━━ 180s 1

/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 126 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



✅ Best Embedding (Inference) model saved as 'best_vehicle_reid_embedding_model.h5'

--- Evaluating Model on Test Set (Using Embeddings) ---
Evaluating model on test pairs...


Evaluating: 100%|██████████| 106/106 [00:57<00:00,  1.83it/s]



📊 --- Evaluation Summary ---
✅ Total pairs tested: 106 (Pos: 53, Neg: 53)
✅ Accuracy: 98.11%
✅ Precision: 1.0000
✅ Recall: 0.9623
✅ F1 Score: 0.9808
✅ Average Positive Distance: 0.0464
✅ Average Negative Distance: 0.2564
✅ Optimal Distance Threshold: 0.0948
✅ AUC: 0.9964
✅ Average SSIM for Positive Pairs: 0.1258
✅ Average SSIM for Negative Pairs: 0.1009
✅ ROC curve saved as 'roc_curve.png'

✅ Test results saved to 'test_results.json'
Cleaning up temporary directories...

🎉 Classification-based Training completed successfully!
Test Accuracy: 98.11%
Test AUC: 0.9964
Test Average SSIM for Positive Pairs: 0.1258
Test Average SSIM for Negative Pairs: 0.1009

--- All Done ---
